# 00. Setup del Proyecto

Este notebook centraliza la validación del entorno antes de ejecutar el resto del pipeline.

## Flujo recomendado
1. Ejecutar este notebook completo.
2. Continuar con `01_download_benchmarks.ipynb`.
3. Ejecutar `02_baseline_audit.ipynb`.
4. Analizar resultados en `03_baseline_analysis.ipynb`.
5. Construir dataset en `04_sft_dataset_curation.ipynb`.
6. Entrenar en `05_qlora_sft.ipynb`.
7. Evaluar impacto en `06_impact_evaluation.ipynb`.
8. Analizar resultados en `07_impact_analysis.ipynb`.

In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import os
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve()

# ── Configuración del proyecto ───────────────────────────────────────────────
SEED          = 42
BASE_MODEL    = "meta-llama/Llama-3.1-8B"
TRAIN_FILE    = "data/processed/sft_train.jsonl"
VALID_FILE    = "data/processed/sft_valid.jsonl"
METADATA_FILE = "data/processed/sft_metadata.csv"

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
OUTPUT_DIR   = PROJECT_ROOT / "outputs"
DATA_DIR     = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/cancio/Escritorio/MUIIA/TFM


In [2]:
def package_version(package_name: str) -> str:
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return "missing"


def build_path_report(root: Path) -> pd.DataFrame:
    tracked_paths = [
        ("Notebooks",         NOTEBOOK_DIR),
        ("Outputs",           OUTPUT_DIR),
        ("Processed train",   root / TRAIN_FILE),
        ("Processed valid",   root / VALID_FILE),
        ("SFT metadata",      root / METADATA_FILE),
        ("CrowS-Pairs cache", root / "data/raw/hf_datasets/crows_pairs_test"),
        ("BBQ cache",         root / "data/raw/hf_datasets/bbq_test"),
    ]
    rows = []
    for label, path in tracked_paths:
        rows.append({
            "Item":   label,
            "Path":   str(path.relative_to(root) if str(path).startswith(str(root)) else path),
            "Exists": path.exists(),
            "Kind":   "dir" if path.is_dir() else "file",
        })
    return pd.DataFrame(rows)

In [3]:
core_packages = [
    "torch",
    "transformers",
    "datasets",
    "accelerate",
    "peft",
    "trl",
    "bitsandbytes",
    "pandas",
    "openpyxl",
    "numpy",
    "pyyaml",
    "langchain",
    "langchain-openai",
    "langchain-ollama",
    "matplotlib",
    "seaborn",
    "jupyter",
    "ipykernel",
]

package_report = pd.DataFrame(
    [{"package": package, "version": package_version(package)} for package in core_packages]
)
package_report

,package,version
0,torch,2.10.0
1,transformers,5.3.0
2,datasets,2.21.0
3,accelerate,1.13.0
4,peft,0.18.1
5,trl,0.15.2
6,bitsandbytes,0.49.2
7,pandas,3.0.1
8,openpyxl,3.1.5
9,numpy,2.4.3


In [4]:
config_summary = pd.DataFrame([
    {"Parámetro": "BASE_MODEL",    "Valor": BASE_MODEL},
    {"Parámetro": "SEED",          "Valor": SEED},
    {"Parámetro": "TRAIN_FILE",    "Valor": TRAIN_FILE},
    {"Parámetro": "VALID_FILE",    "Valor": VALID_FILE},
    {"Parámetro": "METADATA_FILE", "Valor": METADATA_FILE},
])
path_report = build_path_report(PROJECT_ROOT)
env_report = pd.DataFrame(
    [
        {"variable": "OPENROUTER_API_KEY", "configured": bool(os.getenv("OPENROUTER_API_KEY"))},
        {"variable": "HF_TOKEN",           "configured": bool(os.getenv("HF_TOKEN"))},
    ]
)

core_packages = [
    "torch", "transformers", "datasets", "accelerate", "peft", "trl",
    "bitsandbytes", "pandas", "openpyxl", "numpy", "langchain",
    "langchain-openai", "langchain-ollama", "matplotlib", "seaborn",
    "jupyter", "ipykernel",
]
package_report = pd.DataFrame(
    [{"package": p, "version": package_version(p)} for p in core_packages]
)

print("Configuración del proyecto")
display(config_summary)
print("Estado de rutas clave")
display(path_report)
print("Variables de entorno críticas")
display(env_report)
print("Paquetes instalados")
display(package_report)

missing_packages = package_report.loc[package_report["version"] == "missing", "package"].tolist()
missing_paths    = path_report.loc[~path_report["Exists"], "Item"].tolist()
print("Checklist final")
print(f"- Paquetes pendientes: {missing_packages or 'ninguno'}")
print(f"- Rutas pendientes:    {missing_paths or 'ninguna'}")
print("- Siguiente notebook: notebooks/01_download_benchmarks.ipynb")

Configuración del proyecto


,Parámetro,Valor
0,BASE_MODEL,meta-llama/Llama-3.1-8B
1,SEED,42
2,TRAIN_FILE,data/processed/sft_train.jsonl
3,VALID_FILE,data/processed/sft_valid.jsonl
4,METADATA_FILE,data/processed/sft_metadata.csv


Estado de rutas clave


,Item,Path,Exists,Kind
0,Notebooks,notebooks,True,dir
1,Outputs,outputs,True,dir
2,Processed train,data/processed/sft_train.jsonl,True,file
3,Processed valid,data/processed/sft_valid.jsonl,True,file
4,SFT metadata,data/processed/sft_metadata.csv,True,file
5,CrowS-Pairs cache,data/raw/hf_datasets/crows_pairs_test,True,dir
6,BBQ cache,data/raw/hf_datasets/bbq_test,True,dir


Variables de entorno críticas


,variable,configured
0,OPENROUTER_API_KEY,True
1,HF_TOKEN,False


Paquetes instalados


,package,version
0,torch,2.10.0
1,transformers,5.3.0
2,datasets,2.21.0
3,accelerate,1.13.0
4,peft,0.18.1
5,trl,0.15.2
6,bitsandbytes,0.49.2
7,pandas,3.0.1
8,openpyxl,3.1.5
9,numpy,2.4.3


Checklist final
- Paquetes pendientes: ninguno
- Rutas pendientes:    ninguna
- Siguiente notebook: notebooks/01_download_benchmarks.ipynb
